# Step 3 — ComplexHeatmap (R / sc_r env)

## 1. Libraries

In [ ]:
library(ComplexHeatmap)
library(circlize)
library(RColorBrewer)
library(grid)
cat("ComplexHeatmap", as.character(packageVersion("ComplexHeatmap")), "\n")

## 2. Paths

In [ ]:
BASE      <- "/home/kibr/Work/Vas_try"
RESULTS   <- file.path(BASE, "Results")
TOP_GENES <- file.path(RESULTS, "ewce_top_genes.csv")
GENE_CSV  <- file.path(BASE, "Data", "brain_vascular_disease_genes_curated.csv")

setwd(BASE)
cat("Working dir:", getwd(), "\n")
cat("Top genes CSV exists:", file.exists(TOP_GENES), "\n")

## 3. Colour Palettes

Built dynamically from `ewce_top_genes.csv` so they always match the current gene list.

In [ ]:
gene_meta <- read.csv(GENE_CSV, stringsAsFactors=FALSE)
# head(gene_meta)

In [ ]:
df_top <- read.csv(TOP_GENES, stringsAsFactors=FALSE)
# df_top$pathway = gene_meta$pathway[match(df_top$gene_name, gene_meta$gene)]
# df_top

In [ ]:
# df_top <- read.csv(TOP_GENES, stringsAsFactors=FALSE)

# # Use pathway_short if available, otherwise first segment of pathway
# if (!"pathway_short" %in% colnames(df_top)) {
#   df_top$pathway_short <- sub("/.*", "", df_top$pathway)
#   df_top$pathway_short <- trimws(df_top$pathway_short)
# }
# df_top$pathway_short[df_top$pathway_short == ""] <- "Other"

diseases  <- unique(df_top$disease)
# pw_groups <- unique(df_top$pathway_short)
cat("Diseases  :", length(diseases),  "\n")
# cat("PW groups :", length(pw_groups), "\n")

# 15-colour base palette for diseases (tab20-like)
BASE_COLS <- c("#E64B35","#4DBBD5","#00A087","#3C5488","#F39B7F","#8491B4",
               "#91D1C2","#FFDC91","#B09C85","#DC0000","#7E6148","#7CAE00",
               "#C77CFF","#FF7F00","#17BECF","#AEC7E8","#FFBB78","#98DF8A",
               "#FF9896","#C5B0D5")
make_pal <- function(labels, base=BASE_COLS) {
  cols <- colorRampPalette(base)(length(labels))
  setNames(cols, labels)
}

DISEASE_PAL <- make_pal(diseases)
# PATHWAY_PAL <- make_pal(pw_groups,
#                 base=c("#1F77B4","#FF7F0E","#2CA02C","#D62728","#9467BD",
#                        "#8C564B","#E377C2","#7F7F7F","#BCBD22","#17BECF",
#                        "#AEC7E8","#FFBB78","#98DF8A","#FF9896","#C5B0D5"))

# Preview
par(mfrow=c(1,1), mar=c(6,2,2,1))
barplot(rep(1, length(diseases)), col=DISEASE_PAL, names.arg=diseases,
        las=2, cex.names=0.6, main="Disease palette", border=NA)

In [ ]:
df_top

## 4. Data Loading Helper

In [ ]:
load_ct_data <- function(cell_type) {
  csv_path <- file.path(RESULTS, paste0("heatmap_pseudobulk_", cell_type, ".csv"))
  if (!file.exists(csv_path))
    stop(paste("Pseudobulk CSV not found:", csv_path,
               "\nRun the Python Step 3 notebook first."))

  mat      <- as.matrix(read.csv(csv_path, row.names=1, check.names=FALSE))
  genes_ct <- df_top[df_top$cell_type == cell_type, ]
  genes_ct <- genes_ct[!duplicated(genes_ct$gene_name), ]
  rownames(genes_ct) <- genes_ct$gene_name

  # Keep only genes present in both
  genes_in <- intersect(colnames(mat), genes_ct$gene_name)
  mat      <- mat[, genes_in, drop=FALSE]
  genes_ct <- genes_ct[genes_in, , drop=FALSE]

  # adjust the region name
  ID = match(rownames(mat),rg_meta$brain_region)
  rownames(mat) = rg_meta[ID,]$region_name
  
  cat(sprintf("%s: %d regions × %d genes\n",
              cell_type, nrow(mat), ncol(mat)))
  list(mat=mat, meta=genes_ct, genes_in=genes_in)
}
cat("Helper defined.\n")

In [ ]:
rg_meta = read.csv("/home/kibr/Work/Vascular_atlas/Data/2_Vascular_Mapped_Region_ID.csv")
rg_meta = read.csv("/home/kibr/Work/Vascular_atlas/Data/region.csv")
rg_meta$brain_region = paste(rg_meta$regionID,rg_meta$region_abb,sep = "_")
rg_meta$region_name = trimws(stringr::str_to_title(rg_meta$Region_layer_1))
# rg_meta$brain_region[rg_meta$brain_region == "21_Cervical Spinal-Cord"] = "21_Spinal-cord"
# rg_meta

## 5. ComplexHeatmap Builder

In [ ]:
build_ht <- function(cell_type,
                    cluster_rows    = TRUE,
                    cluster_cols    = TRUE,
                    row_split_by    = "disease",   # "disease" | "pathway_short" | NULL
                    show_row_dend   = FALSE,
                    show_col_dend   = TRUE,
                    col_font_size   = 16,
                    row_font_size   = 16,
                    width_cm        = NULL,
                    height_cm       = NULL) {

  d       <- load_ct_data(cell_type)
  mat     <- d$mat
  meta    <- d$meta

  ## Adding region layer colors
  ID = match(rownames(mat),rg_meta$region_name)
  df = rg_meta[ID, c("region_name",'Region_layer_4')]
  df$region_color <- NA
  df[df$Region_layer_4 == "Cerebral Cortex",]$region_color = "#009E73"
  df[df$Region_layer_4 == "Brain Stem and Spinal Cord",]$region_color = "#D55E00"
  df[df$Region_layer_4 == "Cerebellum",]$region_color = "#0072B2"
  df[df$Region_layer_4 == "Subcortical Region",]$region_color = "#56B4E9"
  df[df$Region_layer_4 == "Cerebral Cortex Watershed",]$region_color = "#E69F00"
  df[df$Region_layer_4 == "White Matter",]$region_color = "#CC79A7"
  # df[df$Region_layer_4 == "Olfactory Bulb",]$region_color = "#999999"
  # df[df$Region_layer_4 == "Choroid Plexus",]$region_color = "#F0E442"
  df[df$Region_layer_4 == "Leptomeninges",]$region_color = "#000000"
  df[df$Region_layer_4 == "Major Vessel",]$region_color = "#FF0000"
  # ## Assign region color
  region_color <- df$region_color
  names(region_color) <-df$region_name

  col_fun <- colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B"))

  # Left annotation: Disease + Pathway group
  dis_vals <- meta[rownames(t(mat)), "disease"]
  pw_vals  <- meta[rownames(t(mat)), "pathway_short"]

  # Restrict palette to only values present
  dis_col_used <- DISEASE_PAL[intersect(names(DISEASE_PAL), unique(dis_vals))]
  # pw_col_used  <- PATHWAY_PAL[intersect(names(PATHWAY_PAL), unique(pw_vals))]

  left_ann <- rowAnnotation(
    Disease = dis_vals,
    Pathway = pw_vals,
    col = list(Disease = dis_col_used),#, Pathway = pw_col_used),
    annotation_name_gp = gpar(fontsize=8),
    annotation_legend_param = list(
      Disease = list(title_gp=gpar(fontsize=8), labels_gp=gpar(fontsize=7))#,
      # Pathway = list(title_gp=gpar(fontsize=8), labels_gp=gpar(fontsize=7))
    )
  )

  # annotation for heatmap
  hl <- HeatmapAnnotation(
    region = colnames(t(mat)),
  #   avg_density = avg_vec,
    col = list(
      region = region_color
    ),
    annotation_name_side = "right"
  )
  # Row split
  row_split_vec <- if (!is.null(row_split_by)) meta[rownames(t(mat)), row_split_by] else NULL

  # Auto-size if not specified
  w_cm <- 16
  h_cm <- 4

  Heatmap(
    t(mat),
    name                 = "Z-score",
    col                  = col_fun,
    left_annotation      = left_ann,
    top_annotation       = hl,
    cluster_rows         = cluster_rows,
    cluster_columns      = cluster_cols,
    show_row_dend        = show_row_dend,
    show_column_dend     = show_col_dend,
    row_split            = row_split_vec,
    show_row_names       = TRUE,
    show_column_names    = TRUE,
    row_names_gp         = gpar(fontsize=row_font_size, fontface="italic"),
    # row_names_rot        = 90,
    column_names_gp      = gpar(fontsize=col_font_size),
    column_names_rot     = 90,
    heatmap_legend_param = list(
      title    = "Z-score\n(log₂CPM)",
      title_gp = gpar(fontsize=9),
      labels_gp = gpar(fontsize=8)
    ),
    column_title         = paste0(cell_type,
                                  " — Top EWCE Disease Genes × Brain Region"),
    column_title_gp      = gpar(fontsize=11, fontface="bold")#,
    # width                = unit(w_cm, "cm"),
    # height               = unit(h_cm, "cm")
  )
}
cat("build_ht() defined.\n")

## 6. Interactive Heatmaps

Run each cell to view the heatmap inline. Adjust `build_ht()` parameters as needed.

In [ ]:
# ── Endothelial ───────────────────────────────────────────────────────────────
ht_endo <- build_ht("Endothelial")

draw(ht_endo, merge_legend=TRUE)

In [ ]:
# ── Mural_Cell ────────────────────────────────────────────────────────────────
ht_mural <- build_ht("Mural_Cell")
draw(ht_mural, merge_legend=TRUE)

In [ ]:
# ── Fibroblast ────────────────────────────────────────────────────────────────
ht_fibro <- build_ht("Fibroblast")
draw(ht_fibro, merge_legend=TRUE)

## 7. Customisation Examples

```r
# No row clustering, split by pathway group instead of disease
build_ht("Endothelial", cluster_rows=FALSE, row_split_by="pathway_short")

# Larger fonts / fixed size
build_ht("Mural_Cell", col_font_size=9, row_font_size=10, width_cm=18, height_cm=8)

# No splits
build_ht("Fibroblast", row_split_by=NULL)
```

In [ ]:
# ── Customise here and re-run ─────────────────────────────────────────────────
ht_custom <- build_ht(
  "Endothelial",
  cluster_rows  = TRUE,
  cluster_cols  = TRUE,
  row_split_by  = "disease",    # "disease" | "pathway_short" | NULL
  col_font_size = 7,
  row_font_size = 8
)
draw(ht_custom, merge_legend=TRUE)

## 8. Save PDFs

Runs `build_ht()` fresh for each cell type and saves to `../Results/`.

In [ ]:
for (ct in c("Endothelial", "Mural_Cell", "Fibroblast")) {
  out_pdf <- file.path(RESULTS, paste0("heatmap_", ct, "_complexheatmap.pdf"))
  tryCatch({
    ht <- build_ht(ct)
    pdf(file=out_pdf, width=22, height=7)
    draw(ht, merge_legend=TRUE)
    dev.off()
    cat("Saved:", out_pdf, "\n")
  }, error=function(e) {
    cat("SKIPPED", ct, ":", conditionMessage(e), "\n")
  })
}